# GeoDataCatalog — notebook d'exemple

Ce notebook illustre les cas d'usage principaux de la librairie :
- Chargement du catalogue depuis `catalog.yml`
- Définition d'une zone d'intérêt (AOI)
- Sélection interactive des layers
- Fetch parallèle
- Exploration sur carte
- Workflow STAC (tri, filtre, chargement raster dans `results`)
- Export

## 1. Initialisation

In [ ]:
import logging
import geopandas as gpd
from geodatacatalog import load_catalog, GeoRequest

# Optional: load API keys from .env (see .env.example) for providers that need them
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")

catalog = load_catalog("catalog.yml")

In [ ]:
# Liste de tous les layers disponibles
catalog.info()

## 2. Zone d'intérêt (AOI) et requête

In [ ]:
# Option A — depuis un fichier
# aoi = gpd.read_file("ma_zone.geojson")
# request = GeoRequest(aoi_gdf=aoi, resolution=10)

# Option B — bbox manuelle
request = GeoRequest(
    bbox=(1.35, 43.55, 1.50, 43.65),
    crs="EPSG:3857",
    resolution=30,
    datetime="2024-06-01/2024-08-31",
    cloud_cover=20,
)

request.explore()

## 3. Sélection interactive des layers

Cochez les layers souhaités, puis exécutez la cellule de fetch.

In [ ]:
# Filtres disponibles : kind="RASTER" | "VECTOR", provider="ign"
#sel = catalog.select_layers()

In [ ]:
#sel.get()  # layers cochés

## 4. Fetch

In [ ]:
results = catalog.fetch(request, layers=['earth_search__S2_L2A', 'soilgrids_sand__sand_0_5cm', 'ign__foret', 'ign_wmts__foret'])
results

## 5. Exploration sur carte

In [ ]:
# Tous les layers : vecteurs, rasters (emprise + aperçu), footprints STAC + thumbnails
results.explore()

## 6. Workflow STAC — tri, filtre, chargement dans results

`.load()` sur une `_LayerSelection` met à jour `results[layer_id]` in-place
et retourne `results` — pas besoin de variable intermédiaire.

In [ ]:
# Inspecter les items disponibles (id, date, cloud cover, coverage AOI)
results.sel("earth_search__S2_L2A")

In [ ]:
# Charger le meilleur item (le moins nuageux)
results["earth_search__S2_L2A"] = results.sel("earth_search__S2_L2A").sort("cloud_cover").first().load(bands=["red", "green", "blue", "nir"])['earth_search__S2_L2A']

# results["earth_search__S2_L2A"] est maintenant un xr.Dataset
results

In [ ]:
ds = results["earth_search__S2_L2A"]
ds

In [ ]:
# Calculer un indice spectral (NDVI)
ndvi = (ds.nir - ds.red) / (ds.nir + ds.red)
ndvi.isel(time=0).plot(cmap="RdYlGn", vmin=-0.2, vmax=0.8)

In [ ]:
# Variante : filtrer par date avant de charger
# results.sel("earth_search__S2_L2A")["2024-07-15"].load(bands=["red", "nir"])

## 7. Accès aux autres types de layers

In [ ]:
# Vecteur
results["ign__foret"].head()

In [ ]:
# Raster WMTS avec légende catégorisée
foret = results["ign_wmts__foret"]
print(foret.attrs["legend"])  # {classe: {color, label}}
foret

## 8. Export

In [ ]:
results.sel("ign__foret").export("./output")

## 9. Cache

In [ ]:
catalog.clear_cache("ign__communes")  # un layer
# catalog.clear_cache()                   # tout le cache